In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

PARQUET_PATH = Path("/Users/zhasik/Desktop/krisha/data/index/index.parquet")
PROJECT_ROOT = Path("/Users/zhasik/Desktop/krisha")


In [2]:
df = pd.read_parquet(PARQUET_PATH)
print("shape:", df.shape)

CURRENT_YEAR = 2025
df_feat = df.copy()

df_feat["floor_ratio"] = df_feat["floor"] / df_feat["floors_total"]
df_feat["is_first"] = (df_feat["floor"] == 1).astype(int)
df_feat["is_last"]  = (df_feat["floor"] == df_feat["floors_total"]).astype(int)
df_feat["building_age"] = CURRENT_YEAR - df_feat["year_built"]
df_feat["floor_ratio"] = df_feat["floor_ratio"].clip(0, 1)

TARGET = "log_price_per_m2"

FEATURES_NUM = [
    "area","rooms","floor","floors_total",
    "floor_ratio","is_first","is_last","building_age"
]
FEATURES_CAT = ["district","building_type"]
FEATURES = FEATURES_NUM + FEATURES_CAT

df_feat[FEATURES + [TARGET]].isna().mean().sort_values(ascending=False).head(20)


shape: (1885, 13)


area                0.0
rooms               0.0
floor               0.0
floors_total        0.0
floor_ratio         0.0
is_first            0.0
is_last             0.0
building_age        0.0
district            0.0
building_type       0.0
log_price_per_m2    0.0
dtype: float64

In [3]:
X = df_feat[FEATURES]
y = df_feat[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=df_feat["district"]
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=X_temp["district"]
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# tuning будем делать на train+val
X_tune = pd.concat([X_train, X_val], axis=0)
y_tune = pd.concat([y_train, y_val], axis=0)

# для стратификации в CV используем district
strat_labels = X_tune["district"].astype(str).values


Train: (1319, 10) Val: (283, 10) Test: (283, 10)


In [4]:
# если optuna не установлен:
# !pip install -U optuna

import optuna
from catboost import CatBoostRegressor, Pool


In [5]:
cat_features_idx = [X_tune.columns.get_loc(c) for c in FEATURES_CAT]

def cv_score_catboost(params, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    maes = []
    r2s = []

    for tr_idx, va_idx in skf.split(X_tune, strat_labels):
        X_tr, X_va = X_tune.iloc[tr_idx], X_tune.iloc[va_idx]
        y_tr, y_va = y_tune.iloc[tr_idx], y_tune.iloc[va_idx]

        model = CatBoostRegressor(
            **params,
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=seed,
            verbose=False
        )

        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            cat_features=cat_features_idx
        )

        pred_log = model.predict(X_va)
        ppm2_true = np.exp(y_va.values)
        ppm2_pred = np.exp(pred_log)

        maes.append(mean_absolute_error(ppm2_true, ppm2_pred))
        r2s.append(r2_score(y_va.values, pred_log))

    return float(np.mean(maes)), float(np.mean(r2s))


In [6]:
def objective(trial: optuna.Trial):
    params = {
        "iterations": trial.suggest_int("iterations", 1500, 6000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
        "depth": trial.suggest_int("depth", 6, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 50.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "rsm": trial.suggest_float("rsm", 0.6, 1.0),
        "early_stopping_rounds": 200,
        "allow_writing_files": False,
    }

    mae, r2 = cv_score_catboost(params, n_splits=5, seed=42)
    trial.set_user_attr("cv_r2_log", r2)
    return mae  # минимизируем MAE ₸/м²

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("Best MAE:", study.best_value)
print("Best params:", study.best_params)
print("Best CV R2(log):", study.best_trial.user_attrs["cv_r2_log"])


[I 2026-01-20 14:34:20,881] A new study created in memory with name: no-name-b0dcb06f-e2b3-4aa0-ad76-09c12320b332
[I 2026-01-20 14:34:29,446] Trial 0 finished with value: 115145.62370314736 and parameters: {'iterations': 3955, 'learning_rate': 0.018591241661041116, 'depth': 8, 'l2_leaf_reg': 1.7080362837005743, 'min_data_in_leaf': 8, 'random_strength': 0.07705954612896004, 'bagging_temperature': 1.7935634281633255, 'subsample': 0.8809676575637041, 'rsm': 0.6581836311015631}. Best is trial 0 with value: 115145.62370314736.
[I 2026-01-20 14:34:35,123] Trial 1 finished with value: 116056.97678557858 and parameters: {'iterations': 3059, 'learning_rate': 0.07588522571939348, 'depth': 8, 'l2_leaf_reg': 8.610371929930512, 'min_data_in_leaf': 23, 'random_strength': 1.0925681674962475, 'bagging_temperature': 0.7378874732472043, 'subsample': 0.6049077176846059, 'rsm': 0.6358138128581484}. Best is trial 0 with value: 115145.62370314736.
[I 2026-01-20 14:34:46,319] Trial 2 finished with value: 115

Best MAE: 114153.59568478385
Best params: {'iterations': 4988, 'learning_rate': 0.019132740237319455, 'depth': 7, 'l2_leaf_reg': 2.463294074007148, 'min_data_in_leaf': 34, 'random_strength': 0.447325927893222, 'bagging_temperature': 1.9978470158261286, 'subsample': 0.8878352153731296, 'rsm': 0.765334856988014}
Best CV R2(log): 0.5732782826555496
